# RAGTruth evidence inspection

This notebook displays two fixed RAGTruth records exactly as they appear in the canonical Parquet consumed by the confirmatory experiment. It is a read-only inspection of claims, retrieval-ranked evidence, and masks.


## Reproducible configuration

The RAGTruth confirmatory run is selected from the YAML configuration. The precedence is `NOTEBOOK_CONFIG`, then `notebooks/configs/publichearing_transfer.local.yaml`. The selected run manifest identifies the exact dataset artifact.


In [ ]:
import hashlib
import json
import os
import re
from html import escape
from pathlib import Path

import pandas as pd
import yaml
from IPython.display import HTML, display


def find_repo_root(start=None):
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "notebooks" / "configs" / "publichearing_transfer.example.yaml").is_file():
            return candidate
    raise FileNotFoundError("Repository root with the notebook configuration example was not found.")


def load_ragtruth_run(root):
    requested = os.environ.get("NOTEBOOK_CONFIG")
    config_path = Path(requested) if requested else root / "notebooks" / "configs" / "publichearing_transfer.local.yaml"
    if not config_path.is_absolute():
        config_path = root / config_path
    if not config_path.is_file():
        raise FileNotFoundError(
            "Local notebook configuration not found.\n\n"
            "Copy:\nnotebooks/configs/publichearing_transfer.example.yaml\n\nto:\n"
            "notebooks/configs/publichearing_transfer.local.yaml\n\nand configure the experiment paths."
        )
    raw = yaml.safe_load(config_path.read_text(encoding="utf-8"))
    experiments = raw.get("experiments") if isinstance(raw, dict) else None
    entry = experiments.get("ragtruth_confirmatory") if isinstance(experiments, dict) else None
    value = entry.get("path") if isinstance(entry, dict) else None
    relative_path = Path(value) if isinstance(value, str) else None
    if not relative_path or relative_path.is_absolute() or ".." in relative_path.parts:
        raise ValueError("ragtruth_confirmatory must define a repository-relative path.")
    run_dir = root / relative_path
    if not run_dir.is_dir():
        raise FileNotFoundError(f"RAGTruth confirmatory run not found: {relative_path}")
    return config_path, run_dir


def load_json(path):
    with path.open(encoding="utf-8") as handle:
        return json.load(handle)


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def resolve_recorded_artifact(root, recorded_path):
    recorded = Path(recorded_path)
    if recorded.is_absolute():
        try:
            relative = recorded.relative_to(root)
        except ValueError:
            artifact_roots = ("results", "data")
            artifact_root = next((name for name in artifact_roots if name in recorded.parts), None)
            if artifact_root is None:
                raise ValueError(f"Recorded artifact is outside the repository: {recorded}")
            relative = Path(*recorded.parts[recorded.parts.index(artifact_root):])
    else:
        relative = recorded
    if relative.is_absolute() or ".." in relative.parts:
        raise ValueError(f"Recorded artifact is not repository-relative: {recorded}")
    return root / relative, relative


ROOT = find_repo_root()
os.chdir(ROOT)
config_path, RAGTRUTH_RUN = load_ragtruth_run(ROOT)
try:
    config_label = config_path.relative_to(ROOT)
except ValueError:
    config_label = config_path
print(f"Configuration: {config_label}")


## Selected records

The record identifiers are explicit and stable, rather than selected by row order, timestamp, or random sampling. The pair has one negative and one positive label from the same source, which makes the retrieval order easy to compare.


In [ ]:
EXAMPLE_IDS = (
    "11892:0:162",
    "11894:256:399",
)

run_manifest = load_json(RAGTRUTH_RUN / "manifest.json")
data_metadata = run_manifest["data_audit"]["metadata"]
dataset_path, dataset_relative_path = resolve_recorded_artifact(ROOT, data_metadata["path"])
if not dataset_path.is_file():
    raise FileNotFoundError(f"Canonical RAGTruth dataset not found: {dataset_relative_path}")
actual_sha256 = sha256_file(dataset_path)
expected_sha256 = data_metadata["dataset_sha256"]
if actual_sha256 != expected_sha256:
    raise ValueError("Canonical RAGTruth dataset SHA-256 does not match the confirmatory run manifest.")

columns = [
    "example_id", "source_id", "split", "claim", "label", "evidence_mask",
    "chunk_1", "chunk_1_score", "chunk_2", "chunk_2_score",
    "chunk_3", "chunk_3_score", "chunk_4", "chunk_4_score",
]
dataset = pd.read_parquet(dataset_path, columns=columns)
records = dataset.set_index("example_id", drop=False)
missing = [example_id for example_id in EXAMPLE_IDS if example_id not in records.index]
if missing:
    raise KeyError(f"Requested example_id values are absent from the canonical dataset: {missing}")
examples = records.loc[list(EXAMPLE_IDS)].copy()
if examples["example_id"].duplicated().any():
    raise ValueError("The canonical dataset has duplicate selected example_id values.")

training_view_manifest = load_json(dataset_path.parent / "manifest.json")
top4_run_path, top4_run_relative_path = resolve_recorded_artifact(ROOT, training_view_manifest["parent"]["run_dir"])
top4_manifest = load_json(top4_run_path / "manifest.json")
source_path, source_relative_path = resolve_recorded_artifact(ROOT, top4_manifest["dataset"]["source_path"])
if not source_path.is_file():
    raise FileNotFoundError(f"Original RAGTruth source file not found: {source_relative_path}")
source_sha256 = sha256_file(source_path)
expected_source_sha256 = top4_manifest["dataset"]["source_sha256"]
if source_sha256 != expected_source_sha256:
    raise ValueError("Original RAGTruth source SHA-256 does not match the top-4 run manifest.")

selected_source_ids = set(examples["source_id"].astype(str))
source_rows = {}
with source_path.open(encoding="utf-8") as handle:
    for raw_line in handle:
        source_row = json.loads(raw_line)
        source_id = str(source_row.get("source_id", ""))
        if source_id in selected_source_ids:
            source_rows[source_id] = source_row
missing_source_ids = sorted(selected_source_ids - set(source_rows))
if missing_source_ids:
    raise KeyError(f"Selected source_id values are absent from source_info.jsonl: {missing_source_ids}")

print(f"Dataset: {dataset_relative_path}")
print(f"Dataset SHA-256: {actual_sha256}")
print(f"Original passages: {source_relative_path}")
print(f"Source SHA-256: {source_sha256}")
print(f"Selected records: {', '.join(EXAMPLE_IDS)}")


## Original passages, claim, and retrieval-ranked evidence

The original passages are loaded from `source_info.jsonl` registered in the parent top-4 manifest. They are shown once per source, followed by the selected claims and the chunks that entered the model. Each chunk is shown in the rank stored in the canonical dataset. Its score is retrieval similarity, not a hallucination probability. `evidence_mask` determines whether a slot is valid model input.


In [ ]:
def paragraph(text):
    return escape(str(text)).replace("\n", "<br>")


def original_passages(source_row):
    source_info = source_row.get("source_info", {})
    value = source_info.get("passages", "") if isinstance(source_info, dict) else source_info
    if isinstance(value, list):
        passages = []
        for index, item in enumerate(value):
            text = item.get("text", "") if isinstance(item, dict) else item
            if str(text).strip():
                passages.append((index, str(text).strip()))
        return passages
    source_text = str(value or "")
    marker = re.compile(r"(?:^|\n\s*)passage\s+(\d+)\s*:\s*", re.IGNORECASE)
    matches = list(marker.finditer(source_text))
    if not matches:
        return [(index, text.strip()) for index, text in enumerate(source_text.split("\n\n")) if text.strip()]
    passages = []
    for position, match in enumerate(matches):
        end = matches[position + 1].start() if position + 1 < len(matches) else len(source_text)
        text = source_text[match.end():end].strip()
        if text:
            passages.append((int(match.group(1)) - 1, text))
    return passages


def render_source(source_id, source_row):
    source_info = source_row.get("source_info", {})
    question = source_info.get("question", "") if isinstance(source_info, dict) else ""
    blocks = [
        "<section>",
        f"<h3>Original passages for source {escape(str(source_id))}</h3>",
        f"<p><strong>Question</strong><br>{paragraph(question)}</p>",
    ]
    for passage_index, passage_text in original_passages(source_row):
        blocks.append(f"<p><strong>Passage {passage_index + 1}</strong><br>{paragraph(passage_text)}</p>")
    blocks.append("</section>")
    display(HTML("".join(blocks)))


def render_example(row):
    label = int(row["label"])
    label_text = "positive (hallucination)" if label else "negative (no hallucination)"
    mask = [bool(value) for value in row["evidence_mask"]]
    blocks = [
        "<section>",
        f"<h3>Example {escape(str(row['example_id']))}</h3>",
        f"<p><strong>Source:</strong> {escape(str(row['source_id']))}<br><strong>Split:</strong> {escape(str(row['split']))}<br><strong>Label:</strong> {label} ({label_text})</p>",
        f"<p><strong>Evidence mask:</strong> {escape(str(mask))}</p>",
        f"<p><strong>Claim</strong><br>{paragraph(row['claim'])}</p>",
    ]
    for rank in range(1, 5):
        if not mask[rank - 1]:
            blocks.append(f"<p><strong>Evidence {rank}</strong> - masked slot</p>")
            continue
        score = float(row[f"chunk_{rank}_score"])
        blocks.append(
            f"<p><strong>Evidence {rank}</strong> - retrieval score: {score:.3f}<br>"
            f"{paragraph(row[f'chunk_{rank}'])}</p>"
        )
    blocks.append("</section>")
    display(HTML("".join(blocks)))


for selected_source_id in sorted(selected_source_ids):
    render_source(selected_source_id, source_rows[selected_source_id])

for _, selected_row in examples.iterrows():
    render_example(selected_row)


## Interpretation

The claims, evidence slots, masks, and scores above are read directly from the canonical Parquet. No retrieval, inference, training, sampling, or scientific artifact generation is performed by this notebook.
